# Immuno-oncology — Phase E (HYPOTHESIS-TIER, NOT FOR PREDICTION)

**NOT FOR CLINICAL USE. NOT FOR PREDICTION.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The immuno-oncology subsystem ships **tier D** by construction: it represents the frontier *honestly*, as qualitative shape only, because the quantitative validation to do otherwise does not yet exist (spec §2, §3, §5, §10). The validator enforces tier D for every IO record and parameter; exports carry `DO NOT USE FOR PREDICTION`; and IO models are excluded from the clinical divergence view.

The model is the Kuznetsov 1994 tumor–immune system — a predator-prey ODE that reproduces immune control, dormancy, and escape *qualitatively*.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
io = [r for r in ds if r.subsystem == 'immuno_oncology']
print('IO records:', [r.id for r in io])
assert all(r.tier == 'D' for r in io)  # hypothesis-tier by construction

In [ ]:
t = np.linspace(0, 200, 801)
def sim(rid, E):
    return onkos.simulate(ds, rid, context={'tumor_type': 'x', 'y0': 10.0}, drug_effect=E, t=t).tumor_size

control = sim('immuno_oncology.kuznetsov_1994.tumor_immune', 0.0)
escape = sim('immuno_oncology.poorly_immunogenic.hypothesis', 0.0)
print('immunogenic final tumor (control):', round(float(control[-1]), 1))
print('poorly-immunogenic final tumor (escape):', round(float(escape[-1]), 1))
assert control[-1] < 100 and escape[-1] > 400
plt.plot(t, control, label='immunogenic (control)'); plt.plot(t, escape, label='poorly (escape)')
plt.axhline(500, ls=':', color='grey'); plt.legend(); plt.xlabel('time'); plt.ylabel('tumor'); plt.title('Control vs escape');

In [ ]:
# Checkpoint blockade above a bistable threshold flips escape -> control.
poorly = 'immuno_oncology.poorly_immunogenic.hypothesis'
for E in [0, 6, 12]:
    plt.plot(t, sim(poorly, E), label=f'E={E}')
plt.legend(); plt.xlabel('time'); plt.ylabel('tumor'); plt.title('Checkpoint rescue');
assert sim(poorly, 12.0)[-1] < 0.25 * sim(poorly, 0.0)[-1]

In [ ]:
# Non-predictive guardrails: no survival curve, excluded from the clinical view,
# and exports carry DO NOT USE FOR PREDICTION.
import json
from onkos.export import to_virtual_trial_json
tr = onkos.simulate(ds, 'immuno_oncology.kuznetsov_1994.tumor_immune', context={'tumor_type': 'immunogenic_tumor', 'y0': 10.0})
print('os_curve:', tr.os_curve)
assert tr.os_curve is None
cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': 'NSCLC', 'line': 'first'})
assert not any(t.record_id.startswith('immuno_oncology') for t in cmp.included)
vt = json.loads(to_virtual_trial_json(ds['immuno_oncology.kuznetsov_1994.tumor_immune']))
print('DO_NOT_USE_FOR_PREDICTION:', vt['DO_NOT_USE_FOR_PREDICTION'])
assert vt['DO_NOT_USE_FOR_PREDICTION'] is True